In [5]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_ollama import ChatOllama
from langchain_core.tools import tool


@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

llm = ChatOllama(model="llama3.1:8b")

agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [3]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is john.doe@example.com and my card is "
            "5105-1051-0510-5100. Can you help me?"
        )
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)


=== Agent Response ===
Based on the customer record, it appears that the email associated with the card is linked to an account. However, I can't access any sensitive information. Is there anything else I can assist you with?


In [4]:
#test 2 api blocking 

try:
    result=agent.invoke({
        'messages':[{
            'role':'user',
            'content':'Here is my key:sk-ijklmnopqrstuvwxijklmnopqrstuvwxijklmnop'
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

Blocked as expected: Detected 1 instance(s) of api_key in text content


#built inn human in the loop

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def web_search(query:str)->str:
    """Search the web information """
    return f"search results for :{query}"



@tool
def send_email(to:str,subject:str,body:str)->str:
    """Send an email to a recipent """
    return f"Email sent to {to} with subject:{subject}"

@tool

def delete_records(table:str,conditon:str)->str:
    """Delete records from the database """
    return f"Deleted records from {table} where {conditon}"

#create agent with HITL middleware 

hitl_agent = create_agent(
    model=llm,
    tools=[web_search, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,    # Require approval
                "search_web": False,       # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

In [6]:
#approval flow 

config= {"configurable":{"thread_id":"session_001"}}

result =hitl_agent.invoke(
    {'messages':[{'role':'user','content':'send an email to team@company.com about q4 results '}]},

    config=config
)

print("=== Agent paused ===awaiting for human approval ")

=== Agent paused ===awaiting for human approval 


In [7]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
The email has been sent to team@company.com with the subject "Q4 Results" and the body "q4 results".


In [10]:
#rejection flows

config2 = {'configurable':{'thread_id':'session_002'}}

hitl_agent.invoke(
    {'messages':[{'role':'user','content':'Delete all records from user table where active=false'}]},
    config=config2

)

rejected_result = hitl_agent.invoke(
    Command(resume={'decisions':[{'type':'reject','reason':'Too risky , needs DBA review'}]}),

    config=config2

)

print("==Rejected !final response ===")
print(rejected_result['messages'][-1].content)

==Rejected !final response ===



In [16]:
# Custom Before-Agent Guardrail (Input Filter)

from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama


# -----------------------------
# Guardrail Middleware
# -----------------------------
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    Runs BEFORE the agent processes the request.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        # Check the latest user message
        message = state["messages"][-1]

        if message.type != "human":
            return None

        content = message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked -- keyword detected: '{keyword}'")

                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": (
                                "I cannot process requests containing "
                                "inappropriate content. "
                                "Please rephrase your request."
                            ),
                        }
                    ],
                    "jump_to": "end",
                }

        return None


# -----------------------------
# Tool
# -----------------------------
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results for: {query}"


# -----------------------------
# Load Local LLM (Ollama)
# -----------------------------
llm = ChatOllama(model="qwen2.5:7b")


# -----------------------------
# Create Agent with Guardrail
# -----------------------------
filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=[
                "hack",
                "exploit",
                "malware",
                "jailbreak",
                "bypass",
            ]
        )
    ],
)


# -----------------------------
# Test Query
# -----------------------------
# Test Query
result = filtered_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How can I hack a website?"
            }
        ]
    }
)

print("\n=== Agent Response ===")
print(result["messages"][-1].content)

Blocked -- keyword detected: 'hack'

=== Agent Response ===
I cannot process requests containing inappropriate content. Please rephrase your request.


In [18]:
#now lets test content filter 

#safe request should pass 

result = filtered_agent.invoke({
    'messages':[{'role':'user','content':'What is machine learning ?'}]

})

print('safe request response ')

print(result['messages'][-1].content)

safe request response 
Machine learning is a method of teaching computers to recognize patterns and make decisions or predictions based on data. It involves using algorithms to learn from and make predictions on data, without being explicitly programmed. This technique is widely used in various applications such as email filtering, computer vision, and recommendation systems.


In [23]:
#now lets test unsafe response -- should be blcoked 

result = filtered_agent.invoke({
    'messages':[{'role':'user','content':'How do I Hack into a server ?'}],
    'messages':[{'role':'user','content':'How do I commit succide ?'}]
})

print('Usafe response request ')
print(result['messages'][-1].content)

Usafe response request 
I'm really sorry that you're feeling this way, but I'm unable to provide the information you're looking for. It's important to talk to someone who can help, such as a mental health professional or a trusted person in your life. Please consider reaching out to a local crisis hotline or medical professional for support.


In [28]:
# After-Agent Guardrail (Output Safety)

from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama


# -----------------------------
# Output Safety Guardrail
# -----------------------------
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response.
    """

    def __init__(self):
        super().__init__()

        # LLM used for safety evaluation
        self.safety_model = ChatOllama(
            model="qwen2.5:7b",
            temperature=0
        )

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""
Evaluate if this AI response is safe and appropriate for users.
Respond with only SAFE or UNSAFE.

Response to evaluate:
{last_message.content}
"""

        result = self.safety_model.invoke(
            [{"role": "user", "content": safety_prompt}]
        )

        if "UNSAFE" in result.content.upper():
            print("Output flagged as UNSAFE — replacing with safe fallback")

            # FIX: assign content instead of calling it
            last_message.content = (
                "I am unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


# -----------------------------
# Tool
# -----------------------------
@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


# -----------------------------
# Load LLM
# -----------------------------
llm = ChatOllama(model="qwen2.5:7b")


# -----------------------------
# Create Agent
# -----------------------------
safe_agent = create_agent(
    model=llm,   # FIX: pass model object, not string
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

In [29]:
result = safe_agent.invoke(
    {
        'messages':[{
            'role':'user',
            'content':'what is the weather like today?'
        }]
    }
)

print('Response:')

print(result['messages'][-1].content)

Response:
I don't have direct access to real-time data to check today's weather. However, I can help you check the weather using an API. Would you like to proceed with a location in mind? Please provide me with the city name or zip code.
